# **Caracare Casestudy**

In [0]:
%sql
-- Create dim_patient table
CREATE TABLE IF NOT EXISTS caracare_casestudy.`00_bronze`.dim_patient
USING DELTA
AS
SELECT DISTINCT
    patient_id, -- Unique identifier for each patient
    patient_email, -- Email address of the patient
    CAST(date_of_birth AS DATE) AS date_of_birth -- Patient's date of birth as DATE type
FROM caracare_casestudy.caracare_raw.caracare_raw;

-- Create dim_doctor table
CREATE TABLE IF NOT EXISTS caracare_casestudy.`00_bronze`.dim_doctor
USING DELTA
AS
SELECT DISTINCT
    md5(concat_ws('|', prescribing_doctor, doctor_specialty, doctor_city)) AS doctor_id, -- Unique doctor ID hash
    prescribing_doctor, -- Name of the prescribing doctor
    doctor_specialty, -- Doctor's specialty
    doctor_city -- City where the doctor practices
FROM caracare_casestudy.caracare_raw.caracare_raw;

-- Create dim_insurance table
CREATE TABLE IF NOT EXISTS caracare_casestudy.`00_bronze`.dim_insurance
USING DELTA
AS
SELECT DISTINCT
    md5(concat_ws('|', insurance_name, insurance_type)) AS insurance_id, -- Unique insurance ID hash
    insurance_name, -- Name of the insurance provider
    insurance_type -- Type of insurance
FROM caracare_casestudy.caracare_raw.caracare_raw;

-- Create fact_prescription table
CREATE TABLE IF NOT EXISTS caracare_casestudy.`00_bronze`.fact_prescription
USING DELTA
AS
SELECT
    prescription_id, -- Unique identifier for each prescription
    patient_id, -- Patient associated with the prescription
    md5(concat_ws('|', prescribing_doctor, doctor_specialty, doctor_city)) AS doctor_id, -- Doctor ID hash
    md5(concat_ws('|', insurance_name, insurance_type)) AS insurance_id, -- Insurance ID hash
    CAST(prescription_start AS DATE) AS prescription_start, -- Prescription start date
    CAST(prescription_end AS DATE) AS prescription_end, -- Prescription end date
    prescription_status, -- Status of the prescription
    CASE WHEN represcription = 'ja' THEN TRUE WHEN represcription = 'nein' THEN FALSE END AS represcription, -- Boolean for represcription
    CAST(represcription_date AS DATE) AS represcription_date -- Date of represcription
FROM caracare_casestudy.caracare_raw.caracare_raw;

-- Create fact_touchpoint table
CREATE TABLE IF NOT EXISTS caracare_casestudy.`00_bronze`.fact_touchpoint
USING DELTA
AS
SELECT
    md5(concat_ws('|', prescription_id, CAST(touchpoint_date AS STRING), touchpoint_type, touchpoint_channel)) AS touchpoint_id, -- Unique touchpoint ID hash
    prescription_id, -- Prescription associated with the touchpoint
    CAST(touchpoint_date AS DATE) AS touchpoint_date, -- Date of the touchpoint
    touchpoint_type, -- Type of touchpoint interaction
    touchpoint_channel, -- Channel of touchpoint interaction
    touchpoint_outcome -- Outcome of the touchpoint
FROM caracare_casestudy.caracare_raw.caracare_raw;